In [1]:
import numpy as np
import scipy.stats as stats
import os

np.set_printoptions(suppress=True, precision=4)

In [2]:
Ker = 1  # Cambiar a 2 para evaluar el modelo NO lineal

if Ker == 1:  # LINEAL
    MC_SVM    = np.array([96.00, 53.75, 98.89, 54.74, 75.00, 71.96, 96.86, 70.72, 90.91])
    OVA_SVM   = np.array([94.67, 56.25, 98.33, 60.80, 73.80, 49.60, 97.20, 76.36, 92.68])
    OVO_SVM   = np.array([98.00, 64.38, 98.33, 63.12, 74.20, 89.02, 97.52, 83.45, 95.58])
    OvA_PSVM  = np.array([93.33, 54.38, 99.52, 61.26, 72.80, 56.98, 96.54, 71.51, 92.68])
    OvO_PSVM  = np.array([98.00, 57.50, 98.33, 64.94, 74.40, 89.21, 97.20, 82.40, 96.02])
    OvA_CPSVM = np.array([96.00, 58.75, 98.89, 61.23, 75.60, 57.75, 96.87, 72.43, 93.33])
    OvO_CPSVM = np.array([98.00, 59.38, 98.30, 64.48, 75.60, 85.80, 97.20, 82.53, 96.41])

else:         # NO LINEAL
    MC_SVM    = np.array([97.33, 85.00, 98.89, 73.77, 75.80, 98.86, 97.36, 85.02, 98.33])
    OVA_SVM   = np.array([97.33, 84.38, 98.33, 73.74, 74.00, 99.24, 97.36, 85.55, 97.50])
    OVO_SVM   = np.array([98.00, 85.00, 98.89, 73.70, 74.60, 99.27, 97.53, 86.07, 97.40])
    OvA_PSVM  = np.array([98.67, 85.00, 100.00, 74.22, 74.40, 99.43, 97.36, 86.73, 97.58])
    OvO_PSVM  = np.array([98.67, 85.63, 99.44, 74.76, 74.40, 99.62, 97.53, 86.74, 97.92])
    OvA_CPSVM = np.array([97.33, 84.38, 100.00, 75.63, 75.60, 99.81, 97.53, 86.99, 97.84])
    OvO_CPSVM = np.array([98.67, 84.38, 100.00, 74.76, 75.40, 99.81, 97.53, 87.12, 98.14])

DAT = np.column_stack((
    MC_SVM, OVA_SVM, OVO_SVM, OvA_PSVM, OvO_PSVM, OvA_CPSVM, OvO_CPSVM
))

DATt = [
    'MC-SVM', 'OVA-SVM', 'OVO-SVM', 'OvA-PSVM', 'OvO-PSVM', 'OvA-CPSVM', 'OvO-CPSVM'
]

nd, nc = DAT.shape  # Número de datasets (nd) y clasificadores (nc)

In [3]:
rk = np.zeros_like(DAT)
for ii in range(nd):
    rk[ii, :] = stats.rankdata(-DAT[ii, :])

print("DAT Matrix:")
print(DAT)
print("\nRanking Matrix:")
print(rk)

# Rango Medio
Mrank = np.mean(rk, axis=0)
print("\nAverage Rank:")
print(Mrank)

# Definir el nombre del archivo según el tipo de kernel
filename_rank = "Ranking_Promedio_ACCU_lin.txt" if Ker == 1 else "Ranking_Promedio_ACCU_no_lin.txt"

# Crear el contenido uniendo cada modelo con su respectivo rango medio
rank_text = ""
for name, rank in zip(DATt, Mrank):
    rank_text += f"{name},{rank:.4f}\n"

# Guardar el archivo de texto
with open(filename_rank, "w", encoding="utf-8") as fid_rank:
    fid_rank.write(rank_text)
    
print(f"\n\nRankings promedio guardados exitosamente como '{filename_rank}'")

DAT Matrix:
[[96.   94.67 98.   93.33 98.   96.   98.  ]
 [53.75 56.25 64.38 54.38 57.5  58.75 59.38]
 [98.89 98.33 98.33 99.52 98.33 98.89 98.3 ]
 [54.74 60.8  63.12 61.26 64.94 61.23 64.48]
 [75.   73.8  74.2  72.8  74.4  75.6  75.6 ]
 [71.96 49.6  89.02 56.98 89.21 57.75 85.8 ]
 [96.86 97.2  97.52 96.54 97.2  96.87 97.2 ]
 [70.72 76.36 83.45 71.51 82.4  72.43 82.53]
 [90.91 92.68 95.58 92.68 96.02 93.33 96.41]]

Ranking Matrix:
[[4.5 6.  2.  7.  2.  4.5 2. ]
 [7.  5.  1.  6.  4.  3.  2. ]
 [2.5 5.  5.  1.  5.  2.5 7. ]
 [7.  6.  3.  4.  1.  5.  2. ]
 [3.  6.  5.  7.  4.  1.5 1.5]
 [4.  7.  2.  6.  1.  5.  3. ]
 [6.  3.  1.  7.  3.  5.  3. ]
 [7.  4.  1.  6.  3.  5.  2. ]
 [7.  5.5 3.  5.5 2.  4.  1. ]]

Average Rank:
[5.3333 5.2778 2.5556 5.5    2.7778 3.9444 2.6111]


Rankings promedio guardados exitosamente como 'Ranking_Promedio_ACCU_lin.txt'


In [4]:
print("\nTest:")

# Chi-cuadrado de Friedman
chi = (12 * nd / (nc * (nc + 1))) * (np.sum(Mrank**2) - (nc * (nc + 1)**2) / 4)
print(f"Chi-square (con {nc-1} grados de libertad): {chi:f}")

# Extensión Iman-Davenport
Friedman_test = (nd - 1) * chi / (nd * (nc - 1) - chi)
print(f"Friedman_test (F_F): {Friedman_test:f}")

# Valor crítico de F
alpha = 0.05
# stats.f.ppf es el equivalente de finv en MATLAB
df1 = nc - 1
df2 = (nc - 1) * (nd - 1)
F_crit = stats.f.ppf(1 - alpha, df1, df2)

print(f"\nValor crítico de F (con una significancia de {alpha:.2f}, {df1} y {df2} grados de libertad): {F_crit:f}\n")

print("Result Test")
print("-----------")
if F_crit > Friedman_test:
    print("H0 : No hay diferencia estadística entre los clasificadores en términos de ACCU.")
else:
    print("H1 : Existen diferencias estadísticas entre los clasificadores en términos de ACCU.")
    print("es decir, la hipótesis nula (H0) de la prueba de Friedman debe ser rechazada.")


Test:
Chi-square (con 6 grados de libertad): 21.547619
Friedman_test (F_F): 5.311812

Valor crítico de F (con una significancia de 0.05, 6 y 48 grados de libertad): 2.294601

Result Test
-----------
H1 : Existen diferencias estadísticas entre los clasificadores en términos de ACCU.
es decir, la hipótesis nula (H0) de la prueba de Friedman debe ser rechazada.


In [5]:
q_alpha = 2.949 # Para alpha=0.05 y nc=7 modelos, q_alpha es aprox 2.949 según la tabla de Nemenyi.
CD = np.sqrt((nc * (nc + 1)) / (6 * nd)) * q_alpha

print(f"\nPrueba de Nemenyi: CD= {CD:f}")

print("\nComparaciones uno a uno:")
for ii in range(nc):
    for jj in range(ii + 1, nc):
        diff = abs(Mrank[ii] - Mrank[jj])
        if diff > CD:
            print(f" {diff} > {CD}")
            print(f"H1 : Existen diferencias estadísticas entre {DATt[ii]} y {DATt[jj]}.\n")
        else:
            print(f" {diff} < {CD}")
            print(f"H0 : No hay diferencia estadística entre {DATt[ii]} y {DATt[jj]}.\n")


Prueba de Nemenyi: CD= 3.003115

Comparaciones uno a uno:
 0.05555555555555536 < 3.003114605427727
H0 : No hay diferencia estadística entre MC-SVM y OVA-SVM.

 2.7777777777777777 < 3.003114605427727
H0 : No hay diferencia estadística entre MC-SVM y OVO-SVM.

 0.16666666666666696 < 3.003114605427727
H0 : No hay diferencia estadística entre MC-SVM y OvA-PSVM.

 2.5555555555555554 < 3.003114605427727
H0 : No hay diferencia estadística entre MC-SVM y OvO-PSVM.

 1.3888888888888884 < 3.003114605427727
H0 : No hay diferencia estadística entre MC-SVM y OvA-CPSVM.

 2.722222222222222 < 3.003114605427727
H0 : No hay diferencia estadística entre MC-SVM y OvO-CPSVM.

 2.7222222222222223 < 3.003114605427727
H0 : No hay diferencia estadística entre OVA-SVM y OVO-SVM.

 0.22222222222222232 < 3.003114605427727
H0 : No hay diferencia estadística entre OVA-SVM y OvA-PSVM.

 2.5 < 3.003114605427727
H0 : No hay diferencia estadística entre OVA-SVM y OvO-PSVM.

 1.333333333333333 < 3.003114605427727
H0 :

In [6]:
meanACCU = np.nanmean(DAT, axis=0)
stdACCU = np.nanstd(DAT, axis=0, ddof=1)
avgRank = np.nanmean(rk, axis=0)

# Orden ascendente por ranking
order = np.argsort(avgRank)

def escape_underscore(s):
    return s.replace("_", "\\_")

header = (
    "\\begin{table}[t]\n"
    "\\centering\n"
    "\\small\n"
    "\\caption{Resumen de desempeño promedio: media, desviación típica y rango medio del ACCU (mayor es mejor) sobre los seis conjuntos de datos.}\n"
    "\\label{tab:accu_summary}\n"
    "\\begin{tabular}{lccc}\n"
    "\\toprule\n"
    "Modelo & $\\overline{\\mathrm{ACCU}}$ & SD & Ranking Promedio \\\\\n"
    "\\midrule\n"
)

rows = ""
for j in order:
    name = escape_underscore(DATt[j])
    rows += f"{name} & {meanACCU[j]:.2f} & {stdACCU[j]:.2f} & {avgRank[j]:.2f} \\\\\n"

footer = (
    "\\bottomrule\n"
    "\\end{tabular}\n"
    "\\end{table}\n"
)

latexTable = header + rows + footer

print("\n--- LaTeX table (ACCU summary) ---")
print(latexTable)

# Guardar a archivo .tex
filename = "table_accu_Multi.tex" if Ker == 1 else "table_accu_Multi_Nonl.tex"
with open(filename, "w", encoding="utf-8") as fid:
    fid.write(latexTable)
    
print(f"Tabla guardada exitosamente como '{filename}'")


--- LaTeX table (ACCU summary) ---
\begin{table}[t]
\centering
\small
\caption{Resumen de desempeño promedio: media, desviación típica y rango medio del ACCU (mayor es mejor) sobre los seis conjuntos de datos.}
\label{tab:accu_summary}
\begin{tabular}{lccc}
\toprule
Modelo & $\overline{\mathrm{ACCU}}$ & SD & Ranking Promedio \\
\midrule
OVO-SVM & 84.84 & 14.38 & 2.56 \\
OvO-CPSVM & 84.19 & 14.96 & 2.61 \\
OvO-PSVM & 84.22 & 15.44 & 2.78 \\
OvA-CPSVM & 78.98 & 17.47 & 3.94 \\
OVA-SVM & 77.74 & 18.94 & 5.28 \\
MC-SVM & 78.76 & 17.70 & 5.33 \\
OvA-PSVM & 77.67 & 18.05 & 5.50 \\
\bottomrule
\end{tabular}
\end{table}

Tabla guardada exitosamente como 'table_accu_Multi.tex'
